# A-share 与 Crypto：已发布研究证据演示

## 目标

这份 Notebook 是**已发布研究证据的只读演示**，不属于实盘，也不证明真实收益。
所有状态、指标和资金曲线都来自显式选定的已发布运行、报告及账本。
它不启动规划、原始数据扫描、物化、训练或报告发布。

Choice 是正式财务 PIT 来源。接口开发与合成验证可独立进行；真实财务处理、PIT 物化和研究运行以全部必需 raw 齐备为前提。
`BLOCKED`、`FAILED`、`UNDEFINED` 均保留原始含义。

## 设置

使用已配置的 `quantlab313` kernel 和已可导入的 QRP 包。填写本地项目目录、
现有 profile、明确 `run_id` 和报告 `instance_identity`；不填写任何 raw 路径。
若同一运行有多份报告，先用只读 `inspect` 列出身份，再明确选择，不能按时间自动挑选。

`TARGET_IDS=()` 展示该运行的所有组合；非空时只展示明确列出的目标。
`NAV_STRIDE=1` 保留全部账本资金点；较大值只改变图形采样，并保留区间边界及末点，
不改变正式统计。下方列出实际源点数与显示点数。

In [ ]:
PROJECT_ROOT = None
PROFILE_ID = "a_share_csmar_choice"
RUN_ID = ""
REPORT_IDENTITY = ""
TARGET_IDS = ()
NAV_STRIDE = 1

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.dates import AutoDateLocator, ConciseDateFormatter
from matplotlib.ticker import StrMethodFormatter

from quant_research_platform.interfaces import queries
from quant_research_platform.profiles.research import research_profile
from quant_research_platform.reports.crypto_hft import (
    REPORT_SCHEMA_ID as CRYPTO_REPORT_SCHEMA,
)
from quant_research_platform.storage.layout import StorageLayout
from quant_research_platform.workflows import research

if PROJECT_ROOT is None or not RUN_ID or not REPORT_IDENTITY:
    raise ValueError("set_explicit_project_profile_run_and_report_identity")

profile = research_profile(PROFILE_ID)
layout = StorageLayout(project_root=Path(PROJECT_ROOT))
run = research.status(profile=profile, layout=layout, run_id=RUN_ID)
document = queries.read_report(
    profile=profile, layout=layout, run_id=RUN_ID, report_identity=REPORT_IDENTITY
)
report = document["report"]
is_crypto = report["schema_id"] == CRYPTO_REPORT_SCHEMA

## 步骤

### 1. 核对运行与报告身份

完整机器报告和已发布 Markdown 保留在对应的不可变报告实例中。
本页只挑选明确命名的展示列；其它正式指标仍以完整报告为准。

In [ ]:
display(
    pd.DataFrame(
        [
            {"证据": "profile", "值": PROFILE_ID},
            {"证据": "run_id", "值": RUN_ID},
            {"证据": "report identity", "值": REPORT_IDENTITY},
            {"证据": "source run status", "值": report["source_run_status"]},
            {"证据": "report status", "值": report["report_status"]},
            {"证据": "scenario", "值": report["scenario_id"]},
            {"证据": "window start", "值": run["research_window"]["start"]},
            {"证据": "window end", "值": run["research_window"]["end"]},
            {"证据": "access", "值": report["access_window"]},
        ]
    )
)
display(Markdown("**来源与就绪状态以所选运行及报告证据为准。**"))

### 2. 展示已发布诊断

Crypto 展示已发布日频统计（包括 365 日年化口径与不足完整 UTC 日的原因），
A-share 展示已发布成本和风险诊断。所有候选与未定义原因均保留；不重算这些数值。

In [ ]:
metric_rows = []
for name in ("daily statistics",) if is_crypto else ("cost", "risk"):
    section = report["sections"][name]
    if section.get("reason") == "not_requested":
        display(pd.DataFrame([{"section": name, **section}]))
    else:
        metric_rows.extend(section["results"])
        if is_crypto:
            display(pd.DataFrame([section["policy"]]))

if metric_rows:
    metrics = pd.DataFrame(metric_rows)
    columns = [
        name
        for name in (
            "candidate_id",
            "metric_id",
            "value",
            "status",
            "reason",
            "observation_count",
        )
        if name in metrics.columns
    ]
    display(Markdown(metrics.loc[:, columns].to_html(index=False, na_rep="—")))
else:
    display(Markdown("未提供该组指标；请查看下方已发布状态与缺失原因。"))

issues = report["sections"]["readiness/blockers"]["issues"]
if issues:
    display(pd.DataFrame(issues))
else:
    display(Markdown("这份已发布报告没有记录额外 blocker；不代表 Choice raw 已齐备。"))

### 3. 从精确账本读取 NAV

下面只聚合账本同一估值时点的资金项，不返回证券持仓或逐笔交易行。
`before / after / end` 保留费用前后及期末估值，图形不会把交易成本跳变抹掉。
Crypto 的 OOS 边界来自账本诊断；空档只作清晰区分，不混称为模型 OOS 收益。

In [ ]:
portfolio_targets = [
    row["target_id"]
    for row in run["target_results"]
    if row["producer_kind"] == "PORTFOLIO"
]
selected_targets = list(TARGET_IDS) if TARGET_IDS else portfolio_targets
if len(set(selected_targets)) != len(selected_targets) or not set(
    selected_targets
).issubset(portfolio_targets):
    raise ValueError("selected_portfolio_target_not_in_run")

curves = [
    queries.portfolio_curve(
        profile=profile,
        layout=layout,
        run_id=RUN_ID,
        target_id=target_id,
        stride=NAV_STRIDE,
    )
    for target_id in selected_targets
]
display(
    pd.DataFrame(
        [
            {
                key: curve[key]
                for key in (
                    "target_id",
                    "status",
                    "reason",
                    "valuation_currency",
                    "execution_mode",
                    "episode_count",
                    "source_point_count",
                    "displayed_point_count",
                    "stride",
                )
                if key in curve
            }
            for curve in curves
        ]
    )
)

### 4. 资金曲线与既有对照

图中使用账本的原始计价币种。Crypto `maker_baseline` 是固定报价诊断对照，
不冒充经验校准策略；A-share 候选按已发布 `candidate_id` 标识，不补造缺失基准收益。
绿色背景是记录中的 OOS 窗口，灰色背景是其间空档。
独立段分别展示，不跨缺口连接；截断段只标注账本实际记录的完整 OOS 边界。

In [ ]:
available = [
    {**curve, **part}
    for curve in curves
    if curve["status"] == "DEFINED"
    for part in (
        curve["episodes"] if curve["execution_mode"] == "independent_segments" else [{}]
    )
]
if available:
    column_count = min(2, len(available))
    row_count = (len(available) + column_count - 1) // column_count
    figure, axes = plt.subplots(
        row_count, column_count, figsize=(14, 3.2 * row_count), squeeze=False
    )
    figure.suptitle("Published research evidence — not live returns", fontsize=12)
    for axis, curve in zip(axes.flat, available, strict=False):
        nav = pd.DataFrame(curve["points"])
        times = pd.to_datetime(nav["timestamp_ns"], unit="ns", utc=is_crypto)
        axis.plot(times, nav["nav"], linewidth=1.3, color="#1f5f8b")
        starts = {
            row["window_index"]: pd.to_datetime(
                row["timestamp_ns"], unit="ns", utc=True
            )
            for row in curve["boundaries"]
            if row["kind"] == "oos_window_start"
        }
        ends = {
            row["window_index"]: pd.to_datetime(
                row["timestamp_ns"], unit="ns", utc=True
            )
            for row in curve["boundaries"]
            if row["kind"] == "oos_window_end"
        }
        if starts:
            indices = sorted(starts.keys() & ends.keys())
            gap_labeled = False
            for previous, following in zip(indices, indices[1:], strict=False):
                if ends[previous] < starts[following]:
                    axis.axvspan(
                        ends[previous],
                        starts[following],
                        color="#e1e3e6",
                        alpha=0.5,
                        label=None if gap_labeled else "Gap",
                    )
                    gap_labeled = True
            for index in indices:
                axis.axvspan(
                    starts[index],
                    ends[index],
                    color="#a7d8be",
                    alpha=0.45,
                    label="OOS" if index == indices[0] else None,
                )
            axis.legend(loc="best", fontsize=8)
        axis.set_title(
            f"{curve['candidate_id']} / {curve['episode_id'][:10]}", fontsize=9
        )
        axis.set_ylabel(f"NAV ({curve['valuation_currency']})")
        axis.yaxis.set_major_formatter(StrMethodFormatter("{x:,.4f}"))
        axis.set_xlabel("Time (UTC)" if is_crypto else "Trading date")
        locator = AutoDateLocator(minticks=3, maxticks=5)
        axis.xaxis.set_major_locator(locator)
        axis.xaxis.set_major_formatter(ConciseDateFormatter(locator))
        axis.grid(alpha=0.2)
    for axis in list(axes.flat)[len(available) :]:
        axis.set_visible(False)
    figure.tight_layout(rect=(0, 0, 1, 0.985))
    display(figure)
    plt.close(figure)
else:
    display(Markdown("没有可读取的已发布组合曲线；保留上方 BLOCKED/FAILED 原因。"))

## 检查

下表只列出已发布曲线的首末点与身份，便于逐项核对；它不生成新的收益统计。
事件级 NAV 不替代正式等距日频样本，Crypto 的年化不足状态不会因画出曲线而变成有效。

In [ ]:
checks = [
    {
        "candidate_id": curve["candidate_id"],
        "episode_id": curve["episode_id"],
        "instance_identity": curve["instance_identity"],
        "first_nav": curve["points"][0]["nav"],
        "last_nav": curve["points"][-1]["nav"],
        "currency": curve["valuation_currency"],
    }
    for curve in available
]
display(pd.DataFrame(checks))
display(
    Markdown(
        "本页读取已发布证据且没有训练或发布操作。"
        "完整状态、来源和正式指标保留在选定运行及报告；此展示不证明真实收益。"
    )
)

## 下一步

如需切换已发布运行或报告，修改显式参数后从头执行。
缺失证据应由已授权的研究流程另行处理，本 Notebook 不代替它执行。
证据必须区分真实与合成来源。真实数据结论只依据满足 raw-first 前提的真实运行；合成展示、测试与包验收分别证明各自范围。